# 01 — SPY Data Exploration

This notebook explores the SPY (S&P 500 ETF) market data that we use for backtesting
our intraday momentum strategies. We examine:
- Price and volume patterns
- Daily return distributions
- Intraday volatility structure
- Data quality and gaps

> **Note:** This is an experimentation notebook for development purposes.
> The final report is produced separately using Quarto.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy import stats

from intraday_momentum.data.provider import DataProvider

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("Setup complete.")

## 1. Download Data

We use `DataProvider` to fetch daily and minute-resolution SPY data via yfinance.
The provider caches data locally in parquet format for reproducibility.

In [ ]:
provider = DataProvider("SPY", cache_dir="../data_cache")

# Daily data — longer period for overview
daily = provider.get_daily_data("2020-01-01", "2025-05-01")
print(f"Daily data: {len(daily)} rows, {daily.index[0].date()} to {daily.index[-1].date()}")
daily.head()

In [ ]:
# Minute data — yfinance limits to ~60 days for 1m resolution
# We fetch a recent window for intraday analysis
minute = provider.get_data("2026-04-15", "2026-05-04", interval="2m")
print(f"Minute data: {len(minute)} rows")
print(f"Date range: {minute.index[0]} to {minute.index[-1]}")
minute.head()

## 2. Summary Statistics

Basic descriptive statistics for the daily data.

In [ ]:
daily.describe().round(2)

In [ ]:
# Daily returns
daily["Return"] = daily["Close"].pct_change()
daily["Log_Return"] = np.log(daily["Close"] / daily["Close"].shift(1))

print("Daily return statistics:")
print(f"  Mean:     {daily['Return'].mean():.5f} ({daily['Return'].mean() * 252:.2%} annualized)")
print(f"  Std:      {daily['Return'].std():.5f} ({daily['Return'].std() * np.sqrt(252):.2%} annualized)")
print(f"  Skewness: {daily['Return'].skew():.3f}")
print(f"  Kurtosis: {daily['Return'].kurtosis():.3f}")
print(f"  Min:      {daily['Return'].min():.4f}")
print(f"  Max:      {daily['Return'].max():.4f}")

## 3. Price and Volume

SPY closing price with volume bars — an overview of the market regime during our backtest period.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                     gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(daily.index, daily["Close"], color="#1f77b4", linewidth=0.8)
ax1.set_ylabel("Price ($)")
ax1.set_title("SPY — Daily Close Price")
ax1.grid(True, alpha=0.3)

ax2.bar(daily.index, daily["Volume"] / 1e6, color="#2ca02c", alpha=0.6, width=1)
ax2.set_ylabel("Volume (M)")
ax2.set_xlabel("Date")

plt.tight_layout()
plt.show()

## 4. Return Distribution

Histogram of daily returns with a fitted normal distribution overlay.
We expect heavier tails than normal (positive kurtosis).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
returns = daily["Return"].dropna()
axes[0].hist(returns, bins=80, density=True, alpha=0.7, color="#1f77b4", edgecolor="white")
x = np.linspace(returns.min(), returns.max(), 200)
mu, sigma = returns.mean(), returns.std()
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=2, label="Normal fit")
axes[0].set_title("Daily Return Distribution")
axes[0].set_xlabel("Return")
axes[0].legend()

# QQ plot
stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title("QQ Plot vs. Normal")

plt.tight_layout()
plt.show()

## 5. Rolling Volatility

14-day rolling standard deviation of returns — this is the lookback window
used by our strategies for position sizing.

In [ ]:
daily["Vol_14d"] = daily["Return"].rolling(14).std() * np.sqrt(252)
daily["Vol_30d"] = daily["Return"].rolling(30).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily.index, daily["Vol_14d"], label="14-day", alpha=0.8)
ax.plot(daily.index, daily["Vol_30d"], label="30-day", alpha=0.8)
ax.axhline(y=0.15, color="gray", linestyle="--", alpha=0.5, label="15% reference")
ax.set_ylabel("Annualized Volatility")
ax.set_title("SPY — Rolling Volatility")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Intraday Patterns

Average absolute return by time-of-day from minute data. This is relevant because
our strategies use minute-of-day sigma bands — we expect higher volatility at
market open and close.

In [ ]:
minute_copy = minute.copy()
minute_copy["time"] = minute_copy.index.time
minute_copy["abs_return"] = minute_copy["Close"].pct_change().abs()

intraday_vol = minute_copy.groupby("time")["abs_return"].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(intraday_vol)), intraday_vol.values * 100, linewidth=0.8)
ax.set_title("SPY — Average Absolute Return by Time of Day")
ax.set_ylabel("Avg |Return| (%)")
ax.set_xlabel("Minute of Trading Day")

n_minutes = len(intraday_vol)
ticks = [0, n_minutes // 4, n_minutes // 2, 3 * n_minutes // 4, n_minutes - 1]
ax.set_xticks(ticks)
ax.set_xticklabels(["Open", "11:00", "12:30", "14:00", "Close"])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Data Quality Check

Check for missing values, gaps in trading days, and data integrity.

In [ ]:
print("=== Daily Data ===")
print(f"Missing values:\n{daily[['Open','High','Low','Close','Volume']].isnull().sum()}")
print(f"\nTotal trading days: {len(daily)}")
print(f"Expected (~252/year): ~{int((daily.index[-1] - daily.index[0]).days / 365.25 * 252)}")

print(f"\n=== Minute Data ===")
print(f"Missing values:\n{minute[['Open','High','Low','Close','Volume']].isnull().sum()}")
print(f"\nTotal bars: {len(minute)}")
trading_days_minute = minute.index.normalize().nunique()
print(f"Trading days covered: {trading_days_minute}")
print(f"Avg bars per day: {len(minute) / trading_days_minute:.0f}")

## Summary

Key observations:
- SPY daily returns show slight negative skewness and positive excess kurtosis (fat tails)
- Intraday volatility follows a U-shape: highest at open and close, lowest around midday
- This U-shape pattern is exactly what our strategies exploit — the minute-of-day sigma bands
  adapt to the time-varying intraday volatility structure
- The 14-day rolling volatility shows regime changes that affect our position sizing